# Build TCR3d Class I/Class II TCR FASTA files

This Google Colab notebook builds TCR alpha/beta FASTA datasets from the **TCR3d Class I and Class II complex groups**. It uses TCR3d's chain-level annotations, so it does **not** assume that a fixed PDB chain letter is always alpha or beta.

## Core output files

1. `TCR3d_ClassI_TCR_AB.fasta`
2. `TCR3d_ClassII_TCR_AB.fasta`
3. `TCR3d_ClassI_ClassII_TCR_AB.fasta`
4. `TCR3d_All_Complex_Chains.fasta`

The notebook also writes an audit CSV, a summary CSV, and a snapshot of the TCR3d metadata used.


## How the workflow works

- TCR3d's `tcr_chain_data.tsv` supplies the PDB ID, complex class, TCR type, and annotated chain ID.
- Protein sequences come from either your existing all-sequences FASTA or the RCSB PDB FASTA service.
- The notebook matches each TCR3d `(PDB ID, chain ID)` annotation to its full protein sequence.
- Alpha chains receive `expected_chain=A`; beta chains receive `expected_chain=B`.
- The combined TCR FASTA is the recommended input for the ANARCI/CDR and BLAST-versus-MMseqs2 notebooks.

Sources: [TCR3d downloads](https://tcr3d.ibbr.umd.edu/downloads), [TCR3d Class I](https://tcr3d.ibbr.umd.edu/class1), [TCR3d Class II](https://tcr3d.ibbr.umd.edu/class2), and [RCSB PDB FASTA](https://www.rcsb.org/).

> **Reproducibility note:** TCR3d is updated. If your existing all-sequences FASTA came from an earlier snapshot, use the upload mode below. The PDB IDs in that FASTA will restrict the output to your existing dataset.


## 1. Install the small set of required packages


In [ ]:
%pip install -q "biopython==1.85" pandas requests tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 26.1 MB/s eta 0:00:00


## 2. Configuration

Keep `SEQUENCE_SOURCE = "UPLOAD_EXISTING_FASTA"` if you already have an all-sequences FASTA. Change it to `"DOWNLOAD_CURRENT_TCR3D"` to build everything from the current TCR3d Class I/Class II lists.


In [ ]:
from pathlib import Path
from io import StringIO
from collections import defaultdict
from datetime import datetime, timezone
import re
import shutil
import time

import pandas as pd
import requests
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from tqdm.auto import tqdm

# Recommended for your existing dataset:
SEQUENCE_SOURCE = "UPLOAD_EXISTING_FASTA"
# Alternative: SEQUENCE_SOURCE = "DOWNLOAD_CURRENT_TCR3D"

OUTPUT_DIR = Path("/content/TCR3d_FASTA_files")
CACHE_DIR = Path("/content/rcsb_fasta_cache")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

TCR3D_CHAIN_URL = (
    "https://tcr3d.ibbr.umd.edu/static/download/tcr_chain_data.tsv"
)
RCSB_FASTA_URL = "https://www.rcsb.org/fasta/entry/{pdb_id}"

# Leave False for a structure-chain dataset. An optional deduplication
# cell is provided near the end of the notebook.
CREATE_UNIQUE_SEQUENCE_FASTA = False

print("Sequence source:", SEQUENCE_SOURCE)
print("Output directory:", OUTPUT_DIR)


Sequence source: UPLOAD_EXISTING_FASTA
Output directory: /content/TCR3d_FASTA_files


## 3. Download and inspect the TCR3d chain annotations

Only rows labeled `CLASSI` or `CLASSII` and `Alpha` or `Beta` are retained. Standalone TCR structures and other complex categories are excluded.


In [ ]:
chain_df = pd.read_csv(TCR3D_CHAIN_URL, sep="\t")
chain_df.columns = [str(c).strip().lower() for c in chain_df.columns]

required_columns = {
    "pdb_id", "complex_type", "tcr_type",
    "cdr1_chains", "cdr2_chains", "cdr3_chains"
}
missing_columns = required_columns - set(chain_df.columns)
if missing_columns:
    raise ValueError(
        f"TCR3d metadata is missing expected columns: {sorted(missing_columns)}"
    )

chain_df["pdb_id"] = chain_df["pdb_id"].astype(str).str.upper().str.strip()
chain_df["complex_type"] = (
    chain_df["complex_type"].fillna("").astype(str).str.upper().str.strip()
)
chain_df["tcr_type"] = (
    chain_df["tcr_type"].fillna("").astype(str).str.title().str.strip()
)

selected_meta = chain_df[
    chain_df["complex_type"].isin(["CLASSI", "CLASSII"])
    & chain_df["tcr_type"].isin(["Alpha", "Beta"])
].copy()

selected_meta["mhc_class"] = selected_meta["complex_type"].map(
    {"CLASSI": "I", "CLASSII": "II"}
)
selected_meta["expected_chain"] = selected_meta["tcr_type"].map(
    {"Alpha": "A", "Beta": "B"}
)

metadata_snapshot = OUTPUT_DIR / "TCR3d_chain_metadata_snapshot.tsv"
selected_meta.to_csv(metadata_snapshot, sep="\t", index=False)

metadata_summary = (
    selected_meta.groupby(["mhc_class", "tcr_type"])
    .agg(metadata_rows=("pdb_id", "size"), unique_pdbs=("pdb_id", "nunique"))
    .reset_index()
)
display(metadata_summary)
print("Unique Class I/Class II PDB IDs in current metadata:", selected_meta["pdb_id"].nunique())


,mhc_class,tcr_type,metadata_rows,unique_pdbs
0,I,Alpha,291,291
1,I,Beta,297,297
2,II,Alpha,96,96
3,II,Beta,96,96


Unique Class I/Class II PDB IDs in current metadata: 393


## 4. Helper functions

These functions parse RCSB-style headers such as `|Chain D|` and `|Chains A, B|`, expand multi-chain entity records, and match the resulting sequences to the TCR3d annotations.


In [ ]:
def make_session():
    retry = Retry(
        total=5,
        connect=5,
        read=5,
        backoff_factor=0.8,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=("GET",),
    )
    session = requests.Session()
    session.mount("https://", HTTPAdapter(max_retries=retry))
    return session


def normalize_pdb_id(value):
    text = str(value).strip().upper()
    match = re.match(r"^([0-9][A-Z0-9]{3})", text)
    return match.group(1) if match else None


def split_chain_cell(value):
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text or text.lower() in {"nan", "none", "-"}:
        return []
    tokens = re.split(r"[,;/\s]+", text)
    return [token.strip() for token in tokens if token.strip()]


def chains_from_description(description):
    # Standard RCSB headers: |Chain D| or |Chains A, B|
    match = re.search(r"\|Chains?\s+([^|]+)", description, flags=re.IGNORECASE)
    if match:
        chains = []
        for token in match.group(1).split(","):
            token = token.strip()
            if not token:
                continue
            # RCSB may show label_asym_id plus the original author ID,
            # for example F[auth E]. TCR3d uses the author chain ID.
            auth_match = re.search(r"\[auth\s+([^\]]+)\]", token, flags=re.IGNORECASE)
            if auth_match:
                chains.append(auth_match.group(1).strip())
            else:
                chains.append(re.sub(r"\[.*?\]", "", token).strip())
        chains = [chain for chain in chains if chain]
        if chains:
            return chains

    # Fallback for custom headers such as 1AO7_ChainD or 1AO7|chain=D
    fallback_patterns = [
        r"(?:^|[|_])chain[=:_-]?([A-Za-z0-9]+)(?:$|[|_])",
        r"^[0-9][A-Za-z0-9]{3}[_|:-]([A-Za-z0-9]+)(?:$|[|])",
    ]
    for pattern in fallback_patterns:
        fallback = re.search(pattern, description, flags=re.IGNORECASE)
        if fallback:
            return [fallback.group(1)]
    return []


def parse_fasta_records(records):
    expanded = []
    unparsed_headers = []
    for record in records:
        pdb_id = normalize_pdb_id(record.description)
        chains = chains_from_description(record.description)
        sequence = str(record.seq).replace("*", "").upper()
        if not pdb_id or not chains:
            unparsed_headers.append(record.description)
            continue
        for chain_id in chains:
            expanded.append({
                "pdb_id": pdb_id,
                "chain_id": str(chain_id).strip(),
                "sequence": sequence,
                "original_header": record.description,
            })
    return expanded, unparsed_headers


def download_rcsb_entry(pdb_id, session):
    cache_path = CACHE_DIR / f"{pdb_id}.fasta"
    if cache_path.exists():
        text = cache_path.read_text()
    else:
        response = session.get(RCSB_FASTA_URL.format(pdb_id=pdb_id), timeout=60)
        response.raise_for_status()
        text = response.text
        cache_path.write_text(text)
        time.sleep(0.05)
    return list(SeqIO.parse(StringIO(text), "fasta"))


def clean_header_value(value):
    text = str(value) if value is not None else "unknown"
    return re.sub(r"[^A-Za-z0-9_.+/-]+", "_", text.strip()) or "unknown"


def make_record(sequence, pdb_id, mhc_class, chain_id, expected_chain, tcr_type, tcr_name):
    record_id = (
        f"{pdb_id}|mhc_class={mhc_class}|chain={clean_header_value(chain_id)}"
        f"|expected_chain={expected_chain}|tcr_type={tcr_type}"
        f"|tcr_name={clean_header_value(tcr_name)}"
    )
    return SeqRecord(Seq(sequence), id=record_id, description="")


print("Helper functions loaded.")


Helper functions loaded.


## 5. Load the protein sequences

### Recommended: upload your existing all-sequences FASTA

If upload mode is selected, run the next cell and choose exactly one `.fasta`, `.fa`, or `.faa` file. Its PDB IDs define your snapshot.

If download mode is selected, the notebook retrieves all current Class I/Class II entries from RCSB. This can take several minutes.


In [ ]:
if SEQUENCE_SOURCE == "UPLOAD_EXISTING_FASTA":
    from google.colab import files

    uploaded = files.upload()
    fasta_names = [
        name for name in uploaded
        if name.lower().endswith((".fasta", ".fa", ".faa", ".fas"))
    ]
    if len(fasta_names) != 1:
        raise ValueError(
            "Upload exactly one FASTA file. "
            f"Detected FASTA candidates: {fasta_names}"
        )

    input_fasta = Path(fasta_names[0])
    input_records = list(SeqIO.parse(str(input_fasta), "fasta"))
    expanded_chains, unparsed_headers = parse_fasta_records(input_records)
    input_pdbs = {item["pdb_id"] for item in expanded_chains}
    selected_meta = selected_meta[selected_meta["pdb_id"].isin(input_pdbs)].copy()
    print(f"Uploaded records: {len(input_records):,}")
    print(f"Expanded chain records: {len(expanded_chains):,}")
    print(f"PDB IDs parsed from uploaded FASTA: {len(input_pdbs):,}")
    print(f"Headers that could not be parsed: {len(unparsed_headers):,}")

elif SEQUENCE_SOURCE == "DOWNLOAD_CURRENT_TCR3D":
    session = make_session()
    current_pdbs = sorted(selected_meta["pdb_id"].unique())
    input_records = []
    download_failures = []

    for pdb_id in tqdm(current_pdbs, desc="Downloading RCSB FASTA entries"):
        try:
            input_records.extend(download_rcsb_entry(pdb_id, session))
        except Exception as exc:
            download_failures.append({"pdb_id": pdb_id, "error": str(exc)})

    expanded_chains, unparsed_headers = parse_fasta_records(input_records)
    print(f"Downloaded PDB entries requested: {len(current_pdbs):,}")
    print(f"Download failures: {len(download_failures):,}")
    print(f"Expanded chain records: {len(expanded_chains):,}")
    print(f"Headers that could not be parsed: {len(unparsed_headers):,}")

else:
    raise ValueError(
        "SEQUENCE_SOURCE must be UPLOAD_EXISTING_FASTA or DOWNLOAD_CURRENT_TCR3D"
    )

if not expanded_chains:
    raise RuntimeError("No chain sequences were parsed. Check the FASTA headers.")
if selected_meta.empty:
    raise RuntimeError(
        "No uploaded PDB IDs matched TCR3d Class I/Class II annotations."
    )


Saving all_sequences.fasta to all_sequences.fasta
Uploaded records: 1,861
Expanded chain records: 2,999
PDB IDs parsed from uploaded FASTA: 379
Headers that could not be parsed: 0


## 6. Expand TCR3d's chain annotations and match sequences

Some PDB entities represent multiple identical physical chains, such as `Chains A, B`. This notebook expands those entries so each annotated chain receives its own FASTA record.


In [ ]:
# Build a case-normalized sequence lookup by (PDB ID, chain ID).
chain_lookup = {}
sequence_conflicts = []
for item in expanded_chains:
    key = (item["pdb_id"].upper(), item["chain_id"].upper())
    if key in chain_lookup and chain_lookup[key]["sequence"] != item["sequence"]:
        sequence_conflicts.append({
            "pdb_id": key[0],
            "chain_id": key[1],
            "first_header": chain_lookup[key]["original_header"],
            "second_header": item["original_header"],
        })
    else:
        chain_lookup[key] = item

targets = []
for _, row in selected_meta.iterrows():
    annotated_chains = []
    for column in ["cdr1_chains", "cdr2_chains", "cdr3_chains"]:
        annotated_chains.extend(split_chain_cell(row[column]))
    # Preserve order while removing repeated chain IDs across CDR columns.
    annotated_chains = list(dict.fromkeys(annotated_chains))

    for chain_id in annotated_chains:
        targets.append({
            "pdb_id": row["pdb_id"],
            "mhc_class": row["mhc_class"],
            "chain_id": chain_id,
            "expected_chain": row["expected_chain"],
            "tcr_type": row["tcr_type"],
            "tcr_name": row.get("tcr_name", "unknown"),
        })

target_df = pd.DataFrame(targets).drop_duplicates(
    subset=["pdb_id", "chain_id", "expected_chain"]
).reset_index(drop=True)

matched_records = []
audit_rows = []
for _, target in target_df.iterrows():
    key = (target["pdb_id"].upper(), str(target["chain_id"]).upper())
    source = chain_lookup.get(key)
    status = "matched" if source is not None else "missing_sequence"
    sequence = source["sequence"] if source is not None else ""

    audit_rows.append({
        **target.to_dict(),
        "status": status,
        "sequence_length": len(sequence),
        "source_header": source["original_header"] if source else "",
    })

    if source is not None:
        record = make_record(
            sequence=sequence,
            pdb_id=target["pdb_id"],
            mhc_class=target["mhc_class"],
            chain_id=target["chain_id"],
            expected_chain=target["expected_chain"],
            tcr_type=target["tcr_type"],
            tcr_name=target["tcr_name"],
        )
        matched_records.append({
            "record": record,
            "pdb_id": target["pdb_id"],
            "mhc_class": target["mhc_class"],
            "expected_chain": target["expected_chain"],
            "tcr_type": target["tcr_type"],
            "chain_id": target["chain_id"],
        })

audit_df = pd.DataFrame(audit_rows)
display(audit_df["status"].value_counts(dropna=False).rename_axis("status").to_frame("count"))
print("Sequence conflicts for the same PDB/chain key:", len(sequence_conflicts))


,count
status,
matched,1194
missing_sequence,24


Sequence conflicts for the same PDB/chain key: 66


## 7. Write the four FASTA files and audit tables


In [ ]:
class_i_records = [
    item["record"] for item in matched_records if item["mhc_class"] == "I"
]
class_ii_records = [
    item["record"] for item in matched_records if item["mhc_class"] == "II"
]
combined_records = class_i_records + class_ii_records

class_i_path = OUTPUT_DIR / "TCR3d_ClassI_TCR_AB.fasta"
class_ii_path = OUTPUT_DIR / "TCR3d_ClassII_TCR_AB.fasta"
combined_path = OUTPUT_DIR / "TCR3d_ClassI_ClassII_TCR_AB.fasta"
all_chains_path = OUTPUT_DIR / "TCR3d_All_Complex_Chains.fasta"

SeqIO.write(class_i_records, class_i_path, "fasta")
SeqIO.write(class_ii_records, class_ii_path, "fasta")
SeqIO.write(combined_records, combined_path, "fasta")

# Restrict the all-chain control file to the same PDB snapshot used above.
selected_pdbs = set(selected_meta["pdb_id"].unique())
annotation_by_key = {
    (item["pdb_id"].upper(), str(item["chain_id"]).upper()): item
    for item in matched_records
}

all_chain_output_records = []
seen_all_chain_keys = set()
for item in expanded_chains:
    if item["pdb_id"] not in selected_pdbs:
        continue
    key = (item["pdb_id"].upper(), item["chain_id"].upper())
    if key in seen_all_chain_keys:
        continue
    seen_all_chain_keys.add(key)

    annotation = annotation_by_key.get(key)
    mhc_class_values = selected_meta.loc[
        selected_meta["pdb_id"] == item["pdb_id"], "mhc_class"
    ].dropna().unique().tolist()
    mhc_class = mhc_class_values[0] if mhc_class_values else "unknown"
    expected = annotation["expected_chain"] if annotation else "non_tcr_or_unannotated"
    role = annotation["tcr_type"] if annotation else "other_complex_chain"
    record_id = (
        f"{item['pdb_id']}|mhc_class={mhc_class}"
        f"|chain={clean_header_value(item['chain_id'])}"
        f"|expected_chain={expected}|role={role}"
    )
    all_chain_output_records.append(
        SeqRecord(Seq(item["sequence"]), id=record_id, description="")
    )

SeqIO.write(all_chain_output_records, all_chains_path, "fasta")

audit_path = OUTPUT_DIR / "TCR3d_FASTA_build_audit.csv"
audit_df.to_csv(audit_path, index=False)

run_info = pd.DataFrame([{
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "sequence_source": SEQUENCE_SOURCE,
    "selected_pdbs": len(selected_pdbs),
    "class_i_tcr_records": len(class_i_records),
    "class_ii_tcr_records": len(class_ii_records),
    "combined_tcr_records": len(combined_records),
    "all_complex_chain_records": len(all_chain_output_records),
    "missing_tcr_sequences": int((audit_df["status"] != "matched").sum()),
    "unparsed_fasta_headers": len(unparsed_headers),
    "sequence_key_conflicts": len(sequence_conflicts),
}])
summary_path = OUTPUT_DIR / "TCR3d_FASTA_build_summary.csv"
run_info.to_csv(summary_path, index=False)
display(run_info.T.rename(columns={0: "value"}))


,value
created_utc,2026-08-17T15:04:13.878719+00:00
sequence_source,UPLOAD_EXISTING_FASTA
selected_pdbs,378
class_i_tcr_records,898
class_ii_tcr_records,296
combined_tcr_records,1194
all_complex_chain_records,2921
missing_tcr_sequences,24
unparsed_fasta_headers,0
sequence_key_conflicts,66


## 8. Validate alpha/beta coverage and sequence contents

A PDB is marked complete when at least one matched alpha chain and one matched beta chain are present. Multiple TCR copies in one structure may produce more than two TCR records, which is valid for a structure-chain dataset.


In [ ]:
matched_audit = audit_df[audit_df["status"] == "matched"].copy()
coverage_rows = []
for (mhc_class, pdb_id), group in matched_audit.groupby(["mhc_class", "pdb_id"]):
    labels = set(group["expected_chain"])
    coverage_rows.append({
        "mhc_class": mhc_class,
        "pdb_id": pdb_id,
        "has_alpha": "A" in labels,
        "has_beta": "B" in labels,
        "complete_alpha_beta_pair": {"A", "B"}.issubset(labels),
        "matched_tcr_chain_records": len(group),
    })

coverage_df = pd.DataFrame(coverage_rows)
coverage_path = OUTPUT_DIR / "TCR3d_alpha_beta_coverage.csv"
coverage_df.to_csv(coverage_path, index=False)

coverage_summary = (
    coverage_df.groupby("mhc_class")
    .agg(
        pdbs=("pdb_id", "nunique"),
        complete_pairs=("complete_alpha_beta_pair", "sum"),
        alpha_present=("has_alpha", "sum"),
        beta_present=("has_beta", "sum"),
    )
    .reset_index()
)
display(coverage_summary)

allowed_amino_acids = set("ACDEFGHIKLMNPQRSTVWYBXZJUO")
sequence_issues = []
for record in combined_records:
    unexpected = sorted(set(str(record.seq).upper()) - allowed_amino_acids)
    if unexpected or len(record.seq) == 0:
        sequence_issues.append({
            "record_id": record.id,
            "length": len(record.seq),
            "unexpected_characters": "".join(unexpected),
        })

print("Empty or invalid TCR sequences:", len(sequence_issues))
print("Missing annotated TCR chains:", int((audit_df["status"] != "matched").sum()))

if (audit_df["status"] != "matched").any():
    display(audit_df[audit_df["status"] != "matched"].head(20))
if unparsed_headers:
    print("First headers that could not be parsed:")
    for header in unparsed_headers[:10]:
        print(" -", header)


,mhc_class,pdbs,complete_pairs,alpha_present,beta_present
0,I,275,269,269,275
1,II,95,95,95,95


Empty or invalid TCR sequences: 0
Missing annotated TCR chains: 24


,pdb_id,mhc_class,chain_id,expected_chain,tcr_type,tcr_name,status,sequence_length,source_header
688,6R0E,II,D,A,Alpha,F11,missing_sequence,0,
689,6R0E,II,E,B,Beta,F11,missing_sequence,0,
832,7NDQ,I,D,A,Alpha,Gag:02,missing_sequence,0,
833,7NDQ,I,E,B,Beta,Gag:02,missing_sequence,0,
834,7NDT,I,D,A,Alpha,UL40:01,missing_sequence,0,
835,7NDT,I,I,A,Alpha,UL40:01,missing_sequence,0,
836,7NDT,I,E,B,Beta,UL40:01,missing_sequence,0,
837,7NDT,I,J,B,Beta,UL40:01,missing_sequence,0,
838,7NDU,I,D,A,Alpha,Gag:02,missing_sequence,0,
839,7NDU,I,E,B,Beta,Gag:02,missing_sequence,0,


## 9. Optional: write a sequence-deduplicated combined TCR FASTA

The four main files preserve each structure-chain observation. For similarity benchmarking, repeated PDB structures can overweight identical TCR sequences. Set `CREATE_UNIQUE_SEQUENCE_FASTA = True` in the configuration cell if you also want a combined file containing one record per exact amino-acid sequence.


In [ ]:
if CREATE_UNIQUE_SEQUENCE_FASTA:
    unique_by_sequence = {}
    duplicate_members = defaultdict(list)
    for record in combined_records:
        sequence = str(record.seq)
        duplicate_members[sequence].append(record.id)
        unique_by_sequence.setdefault(sequence, record)

    unique_records = list(unique_by_sequence.values())
    unique_path = OUTPUT_DIR / "TCR3d_ClassI_ClassII_TCR_AB_unique_sequences.fasta"
    SeqIO.write(unique_records, unique_path, "fasta")

    duplicate_table = pd.DataFrame([
        {
            "representative_id": unique_by_sequence[sequence].id,
            "sequence_length": len(sequence),
            "record_count": len(ids),
            "member_ids": ";".join(ids),
        }
        for sequence, ids in duplicate_members.items()
    ])
    duplicate_table.to_csv(OUTPUT_DIR / "exact_sequence_duplicate_groups.csv", index=False)
    print(f"Unique exact sequences: {len(unique_records):,} / {len(combined_records):,}")
else:
    print("Skipped. The four main structure-chain FASTA files are unchanged.")


Skipped. The four main structure-chain FASTA files are unchanged.


## 10. Preview the files and download one ZIP bundle


In [ ]:
output_inventory = []
for file_path in sorted(OUTPUT_DIR.iterdir()):
    if file_path.is_file():
        output_inventory.append({
            "file": file_path.name,
            "size_bytes": file_path.stat().st_size,
        })
display(pd.DataFrame(output_inventory))

print("\nFirst four records from the combined TCR FASTA:\n")
for record in list(SeqIO.parse(combined_path, "fasta"))[:4]:
    print(f">{record.id}")
    print(str(record.seq)[:90] + ("..." if len(record.seq) > 90 else ""))

zip_base = Path("/content/TCR3d_FASTA_files_bundle")
zip_path = Path(shutil.make_archive(str(zip_base), "zip", OUTPUT_DIR))
print("\nZIP bundle created:", zip_path)

from google.colab import files
files.download(str(zip_path))


,file,size_bytes
0,TCR3d_All_Complex_Chains.fasta,722812
1,TCR3d_ClassII_TCR_AB.fasta,87827
2,TCR3d_ClassI_ClassII_TCR_AB.fasta,352933
3,TCR3d_ClassI_TCR_AB.fasta,265106
4,TCR3d_FASTA_build_audit.csv,134665
5,TCR3d_FASTA_build_summary.csv,283
6,TCR3d_alpha_beta_coverage.csv,9078
7,TCR3d_chain_metadata_snapshot.tsv,81049



First four records from the combined TCR FASTA:

>1AO7|mhc_class=I|chain=D|expected_chain=A|tcr_type=Alpha|tcr_name=A6
KEVEQNSGPLSVPEGAIASLNCTYSDRGSQSFFWYRQYSGKSPELIMSIYSNGDKEDGRFTAQLNKASQYVSLLIRDSQPSDSATYLCAV...
>1AO7|mhc_class=I|chain=E|expected_chain=B|tcr_type=Beta|tcr_name=A6
NAGVTQTPKFQVLKTGQSMTLQCAQDMNHEYMSWYRQDPGMGLRLIHYSVGAGITDQGEVPNGYNVSRSTTEDFPLRLLSAAPSQTSVYF...
>1BD2|mhc_class=I|chain=D|expected_chain=A|tcr_type=Alpha|tcr_name=B7
QQVKQNSPSLSVQEGRISILNCDYTNSMFDYFLWYKKYPAEGPTFLISISSIKDKNADGRFTVFLNKSAKHLSLHIVPSQPGDSAVYFCA...
>1BD2|mhc_class=I|chain=E|expected_chain=B|tcr_type=Beta|tcr_name=B7
NAGVTQTPKFQVLKTGQSMTLQCAQDMNHEYMSWYRQDPGMGLRLIHYSVGAGITDQGEVPNGYNVSRSTTEDFPLRLLSAAPSQTSVYF...

ZIP bundle created: /content/TCR3d_FASTA_files_bundle.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Which file should go into the analysis notebooks?

Use `TCR3d_ClassI_ClassII_TCR_AB.fasta` as the main input for:

- ANARCI and IMGT numbering
- CDR1/CDR2/CDR3 extraction
- the main BLAST-versus-MMseqs2 comparison

Use `TCR3d_All_Complex_Chains.fasta` separately when you want to test whether the workflow correctly rejects MHC, peptide, beta-2-microglobulin, and other non-TCR chains. Do not combine those negative-control results with the main TCR-only accuracy results.
